#  Week 2, Day 3 — May 27, 2026
## Unit Standardization


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd, numpy as np, matplotlib.pyplot as plt, os, json, warnings
warnings.filterwarnings('ignore')
BASE_DIR = "/content/drive/MyDrive/Ocean_Plastic_Project"
DIRS = {
    "raw":            BASE_DIR + "/data/raw",
    "processed":      BASE_DIR + "/data/processed",
    "metadata":       BASE_DIR + "/data/metadata",
    "notebooks":      BASE_DIR + "/notebooks",
    "visualizations": BASE_DIR + "/visualizations"
}
LAT_MIN, LAT_MAX, LON_MIN, LON_MAX = 5, 30, 65, 95
YEAR_MIN, YEAR_MAX = 2015, 2026

In [ ]:
df_plastic = pd.read_csv(DIRS['processed'] + '/plastic_clean_v1.csv', parse_dates=['Date'])
df_climate = pd.read_csv(DIRS['processed'] + '/climate_clean_v1.csv', parse_dates=['Date'])
print(f'Loaded: plastic={df_plastic.shape}, climate={df_climate.shape}')

## Step 1: Convert Plastic_Weight_kg → kg/km²

In [ ]:
import math

# Area of a 1x1 degree grid cell varies with latitude
# Formula: area_km2 = (111.32 km/deg)^2 × cos(lat_rad)
def grid_cell_area_km2(lat):
    
    lat_rad = math.radians(abs(lat))
    return (111.32 ** 2) * math.cos(lat_rad)

# Apply per-record: use Latitude column
df_plastic_v2 = df_plastic.copy()
df_plastic_v2['Cell_Area_km2'] = df_plastic_v2['Latitude'].apply(grid_cell_area_km2)
df_plastic_v2['Plastic_Density_kg_km2'] = (
    df_plastic_v2['Plastic_Weight_kg'] / df_plastic_v2['Cell_Area_km2']
)

print('Unit conversion: Plastic_Weight_kg → Plastic_Density_kg_km2')
print()
print('Sample (before vs after):')
print(df_plastic_v2[['Latitude','Plastic_Weight_kg','Cell_Area_km2','Plastic_Density_kg_km2']].head(5).round(4).to_string())
print()
print('Plastic_Density_kg_km2 stats:')
print(df_plastic_v2['Plastic_Density_kg_km2'].describe().round(6))

## Step 2: Verify SST is in Celsius

In [ ]:
print('SST (°C) verification:')
print(f'  Min: {df_climate["SST (°C)"].min():.2f}°C')
print(f'  Max: {df_climate["SST (°C)"].max():.2f}°C')
print(f'  Mean: {df_climate["SST (°C)"].mean():.2f}°C')
print()
# Indian Ocean SST should be ~24-32°C range — verify
if 20 <= df_climate['SST (°C)'].mean() <= 35:
    print('  SST range is consistent with tropical Indian Ocean (20–35°C)')
    print('  No conversion needed — already in Celsius')
else:
    print('   SST values outside expected range — check units!')

In [ ]:
# Save
df_plastic_v2.to_csv(DIRS['processed'] + '/plastic_clean_v2.csv', index=False)
print('Saved plastic_clean_v2.csv with Plastic_Density_kg_km2 column ✅')
print(f'  Shape: {df_plastic_v2.shape}')
print(f'  New column: Plastic_Density_kg_km2 (derived from Plastic_Weight_kg / Cell_Area_km2)')